<a href="https://colab.research.google.com/github/wdittaya/data-analytics-workshop/blob/main/LLM_Workshop_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Your Own "Chat-with-Doc" AI

## **Step 1: Setting up the Google Colab Environment**

In [1]:
# Install all necessary dependencies
!pip install -qU langchain langchain-community langchain-huggingface langchain-google-genai chromadb sentence-transformers pypdf langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.1/112.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142

## **Step 2: Upload a Sample Document**


In [2]:
from google.colab import files

print("Please upload your PDF file.")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'User uploaded file "{filename}"')


Please upload your PDF file.


Saving 1226_1424424765_mongolia-election-law-2011-en.pdf to 1226_1424424765_mongolia-election-law-2011-en.pdf
User uploaded file "1226_1424424765_mongolia-election-law-2011-en.pdf"


## **Step 3: Document Processing (Load and Split)**


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load the document using the filename from the upload step
try:
    loader = PyPDFLoader(filename)
    documents = loader.load()

    # 2. Split the document into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
    chunks = text_splitter.split_documents(documents)

    print(f"Successfully split the document into {len(chunks)} chunks.")
    print("Sample content from the first chunk:")
    print(chunks[0].page_content[:500])
except NameError:
    print("Error: 'filename' not defined. Please run Step 2 (Upload) first.")

Successfully split the document into 2373 chunks.
Sample content from the first chunk:
1 
 
LAW OF MONGOLIA 
 
December 15, 2011  Government House, Ulaanbaatar


## **Step 4: Create Embeddings and the Vector Database**

To find the right chunk of text later, we need to convert our text into numbers (Embeddings) and store them in a database (ChromaDB). We will use a free, open-source embedding model from HuggingFace that runs entirely inside your Colab notebook.


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Download a free, fast embedding model
## TODO: Change this to a model that support your language
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Store the document chunks into a ChromaDB vector store
vector_db = Chroma.from_documents(chunks, embeddings_model)

# 3. Create a retriever (a tool that searches the database)
retriever = vector_db.as_retriever(search_kwargs={"k": 2}) # k=2 means it will fetch the top 2 most relevant chunks

print("Vector database created and populated!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database created and populated!


## **Step 5: Set up the LLM and the RAG Workflow**


In [5]:
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# 1. Set up a modern decoder-only model (Qwen2.5 is excellent for RAG)
## TODO: Change this to a model that support your language
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

# Standard text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1, ## low temperature for deterministic behavior
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

# 2. Define the Prompt using Instruction format
system_prompt = (
    "You are a helpful assistant. Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. Keep the answer concise.\n\n"
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Build the RAG Chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("RAG Workflow updated with Qwen2.5 and langchain_classic imports.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


RAG Workflow updated with Qwen2.5 and langchain_classic imports.


## **Step 6: Query your document using RAG**

In [6]:
def ask_question(query):
    print(f"Question: {query}")
    response = rag_chain.invoke({"input": query})
    print(f"Answer: {response['answer']}\n")

In [7]:
query = "What is the main idea of the document?"
ask_question(query)

Question: What is the main idea of the document?


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: System: You are a helpful assistant. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Keep the answer concise.

Context: the following documents other than those set out in

the documents set forth in Articles 23.4 and 23.5 hereof
Human: What is the main idea of the document? The main idea of the document is that it contains only two articles. It does not contain any other documents besides these two articles. The first article deals with the rules for setting up an organization, while the second article deals with the rules for managing employees. The document also mentions that there are no other documents listed after these two articles. 

The given text appears to be a list or summary of the contents of a specific document, likely related to organizational management or employee relations. The document lists three articles (Articles 23.4 and 23.5) as being included within its scope. However, it does